In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import librosa
from pydub import AudioSegment
from IPython.display import Audio
import logging 
import torch
import torch.nn.functional as F
import torchaudio
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import classification_report, f1_score, roc_auc_score

from src.helper import *
from src.model import *
from src.model_sota import *

import warnings
warnings.filterwarnings("ignore")

In [2]:
#set config
DATA_PATH = "data/deep-detect/dataset/"
OUTPUT_PATH = "output/"
PREDS_PATH = "output/preds/"
MODELS_PATH = "models/"
BATCH_SIZE = 16

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BEST_MODEL_PATH = "models/hybrid_audio_classifier_mixup__cross_entropy__adamw__lr_0_0001__epochs_20__bs_16.pth"
BEST_MODEL_CLASS = HybridAudioClassifier(num_classes=2, use_spec_augment=True).to(device)

In [3]:
if not os.path.exists(OUTPUT_PATH):
    os.mkdir(OUTPUT_PATH)
if not os.path.exists(PREDS_PATH):
    os.mkdir(PREDS_PATH)
if not os.path.exists(MODELS_PATH):
    os.mkdir(MODELS_PATH)

In [4]:
# --- Setup logging ---
logging.basicConfig(
    filename=os.path.join(OUTPUT_PATH, f"nb_06_analyzing_model_result_log.log"),
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger("Main")
logger.info(f"starting {BEST_MODEL_PATH} result analysis...")

### Preparing the data loader

In [5]:
logger.info(f"preparing the dataloader...")

train_ds = AudioFolderDataset(os.path.join(DATA_PATH, "training/"))
test_ds = AudioFolderDataset(os.path.join(DATA_PATH, "testing/"))
holdout_ds = AudioFolderDataset(os.path.join(DATA_PATH, "holdout/"))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
holdout_loader = DataLoader(holdout_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

# Inspect one batch
for X, y, path in train_loader:
    logger.info(f"batch input shape: {X.shape}")
    logger.info(f"batch labels shape: {y.shape}")
    break

### Loading the model

In [6]:
logger.info(f"loading the trained model...")

model = BEST_MODEL_CLASS
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))

logger.info(f"trained model loaded")

### Evaluating test predictions

In [ ]:
logger.info(f"generating test predictions...")

model.eval()
raw_outputs, y_true, y_pred, y_probs = [], [], [], []
with torch.no_grad():
    for X_batch, y_batch, path_batch in test_loader:
        X_batch = X_batch.to(device)
        outputs = model(X_batch)  # raw logits

        # Apply softmax along class dimension to get probabilities
        probs = F.softmax(outputs, dim=1)

        # Store outputs and probabilities
        raw_outputs.append(outputs.cpu())
        y_probs.append(probs.cpu())

        # Get predicted class (argmax)
        preds = probs.argmax(dim=1).cpu().numpy()
        y_true.extend(y_batch.numpy())
        y_pred.extend(preds)

raw_outputs = torch.cat(raw_outputs).numpy()
y_probs = torch.cat(y_probs).numpy()

#get score summary
logger.info(f"model test f1-score: {f1_score(y_true, y_pred)}")
logger.info(f"model classification report: \n{classification_report(y_true, y_pred, digits=5)}")

[WARN] Failed to load data/deep-detect/dataset/holdout/audio_02268.wav: Error opening 'data/deep-detect/dataset/holdout/audio_02268.wav': Format not recognised.

[WARN] Failed to load data/deep-detect/dataset/holdout/audio_00183.wav: Error opening 'data/deep-detect/dataset/holdout/audio_00183.wav': Format not recognised.